# External predictions, temporal backtesting, and feature shift

**Goal.** Align external predictions to point-in-time rows, evaluate matured labels, inspect temporal windows, and compute distribution shift. The notebook adds a real scikit-learn t-SNE view of the feature space.

**Audience.** Data/ML scientists, fraud analysts, platform engineers, and researchers.

**Prerequisites.** Python 3.12+, a clean checkout, and the base FraudTwin install. The workflow is deterministic and runs offline; service integrations are deliberately out of scope here.

**Source size.** 1,000–10,000 logical payments. Every section writes only compact summaries, manifests, or fingerprints to a temporary directory.

**Interpretation.** Synthetic evidence demonstrates mechanics and invariants, not production prevalence or model performance guarantees.


## External predictions and temporal backtesting



In [ ]:
from collections import Counter
from pathlib import Path

from fraudtwin.config import load_config
from fraudtwin.generation import generate
from fraudtwin.ml.baseline import PredictionRecord
from fraudtwin.reproducibility import sha256_json

config = load_config(Path("configs/benchmarks/m13-camouflage-v1.yaml"))
data = generate(config, write=False)
dataset = data.require_dataset()
rows = dataset.frame.to_dicts()
print({"run_id": data.run_id, "pit_rows": len(rows)})
assert len(rows) > 500

In [ ]:
predictions = [
    PredictionRecord(
        event_id=r["event_id"],
        prediction_timestamp=r["prediction_time"],
        fraud_score=min(1.0, float(r["amount"]) / 5000.0),
    )
    for r in rows
]
print({"predictions": len(predictions), "target": predictions[0].target})
assert len(predictions) == len(rows)

In [ ]:
targets = {p.target for p in predictions}
row_targets = {("event_id", r["event_id"]) for r in rows}
print({"unique_prediction_targets": len(targets), "pit_targets": len(row_targets)})
assert targets == row_targets

In [ ]:
split_counts = Counter(r["split"] for r in rows)
label_counts = Counter("matured" if r["label"] is not None else "unresolved" for r in rows)
print({"splits": dict(split_counts), "labels": dict(label_counts)})

In [ ]:
matured = [r for r in rows if r["label"] is not None]
print(
    {
        "matured": len(matured),
        "unresolved": len(rows) - len(matured),
        "policy": "exclude_unresolved",
    }
)

In [ ]:
thresholds = [
    {"threshold": t, "flagged": sum(p.fraud_score >= t for p in predictions)}
    for t in (0.25, 0.5, 0.75)
]
print(thresholds)
assert thresholds[0]["flagged"] >= thresholds[-1]["flagged"]

In [ ]:
windows = Counter(r["business_event_time"].date().isoformat() for r in rows)
print({"replay_windows": len(windows), "largest_window": max(windows.values())})
assert windows

In [ ]:
coverage = {
    "pit_rows": len(rows),
    "predictions": len(predictions),
    "matured_labels": len(matured),
    "unresolved_labels": len(rows) - len(matured),
}
print(coverage)
assert coverage["pit_rows"] == coverage["predictions"]

In [ ]:
manifest = {
    "run_id": data.run_id,
    "coverage": coverage,
    "thresholds": thresholds,
    "label_policy": "exclude_unresolved",
}
manifest["fingerprint"] = sha256_json(manifest)
print(manifest)

In [ ]:
assert manifest["fingerprint"] == sha256_json(
    {k: v for k, v in manifest.items() if k != "fingerprint"}
)
print("Scores align by stable event identity; evaluation waits for label maturity.")

### Feature-space analysis

The next cells use the existing ml extra. t-SNE is exploratory: it preserves local neighborhoods, is not a classifier, and must not be interpreted as a causal separation.


In [ ]:
import polars as pl

from fraudtwin.ml.drift import DriftConfig, compare_performance, compare_windows

feature_names = [
    name
    for name in (
        "amount",
        "transaction_count_1h",
        "transaction_amount_24h",
        "device_age_days",
        "customers_per_device_24h",
        "credit_utilization",
    )
    if name in dataset.frame.columns
]
feature_frame = (
    dataset.frame.select(feature_names + ["label", "split"])
    .with_columns(
        [pl.col(name).cast(pl.Float64).fill_null(0.0).alias(name) for name in feature_names]
    )
    .drop_nulls(subset=["split"])
)
print({"features": feature_names, "rows": feature_frame.height})
assert feature_frame.height > 100

In [ ]:
numeric = feature_frame.select(feature_names).to_numpy()
try:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    from sklearn.preprocessing import StandardScaler

    scaled = StandardScaler().fit_transform(numeric)
    pca = PCA(n_components=2, random_state=42).fit_transform(scaled)
    sample_n = min(240, len(scaled))
    perplexity = min(30, max(5, sample_n // 10))
    embedding = TSNE(
        n_components=2,
        random_state=42,
        init="pca",
        learning_rate="auto",
        perplexity=perplexity,
        max_iter=350,
    ).fit_transform(scaled[:sample_n])
    print({"pca_shape": pca.shape, "tsne_shape": embedding.shape, "perplexity": perplexity})
    assert embedding.shape == (sample_n, 2)
except ImportError:
    embedding = numeric[: min(240, len(numeric))]
    print(
        {
            "fallback": "install the optional ml extra for sklearn PCA/t-SNE",
            "shape": embedding.shape,
        }
    )

In [ ]:
reference_rows = rows[: len(rows) // 2]
comparison_rows = rows[len(rows) // 2 :]
drift = compare_windows(
    reference_rows,
    comparison_rows,
    DriftConfig(reference_name="train-window", comparison_name="replay-window", minimum_samples=10),
)
print(
    {"metrics": len(drift.metrics), "alerts": len(drift.alerts), "fingerprint": drift.fingerprint}
)
assert drift.reference_count + drift.comparison_count == len(rows)

In [ ]:
matured_rows = [row for row in rows if row["label"] is not None]
if len(matured_rows) >= 20:
    half = len(matured_rows) // 2
    performance = compare_performance(
        matured_rows[:half],
        matured_rows[half:],
        predictions[:half],
        predictions[half:],
        config=DriftConfig(reference_name="early", comparison_name="late", minimum_samples=10),
    )
    print(
        {
            "performance_metrics": len(performance.performance_metrics),
            "performance_alerts": len(performance.alerts),
        }
    )
else:
    print({"performance": "deferred until labels mature", "matured_rows": len(matured_rows)})

In [ ]:
analysis_manifest = {
    "run_id": data.run_id,
    "feature_names": feature_names,
    "tsne_points": int(embedding.shape[0]),
    "drift_fingerprint": drift.fingerprint,
    "label_policy": "exclude_unresolved",
}
analysis_manifest["fingerprint"] = sha256_json(analysis_manifest)
print(analysis_manifest)
assert analysis_manifest["fingerprint"] == sha256_json(
    {k: v for k, v in analysis_manifest.items() if k != "fingerprint"}
)

## Verification and next step

Re-run the offline cells from a clean checkout and compare the printed fingerprints. For service-backed publication, continue with the relevant integration runbook after this notebook; do not treat synthetic metrics as a deployment SLO.
